# ReAct Agents

**ReAct** (Reasoning + Acting) is the standard pattern for tool-using LLM agents.
The agent alternates between:

1. **Thought** — the model reasons about what to do next
2. **Action** — the model calls a tool
3. **Observation** — the tool result is returned to the model

This loop repeats until the model decides it has enough information to answer.

```
User query
   │
   ▼
┌──────────────────────────────────┐
│  LLM: Thought → Action (tool)   │◄──┐
└──────────────┬───────────────────┘   │
               │ tool call             │
               ▼                       │
         Tool execution                │
               │ Observation           │
               └───────────────────────┘
                        │ stop_reason == end_turn
                        ▼
                  Final answer
```

**What's covered**

| Section | Topic |
|---|---|
| 1 | Setup |
| 2 | First ReAct agent with LangGraph |
| 3 | Defining tools with `@tool` |
| 4 | Multi-tool agent (biomedical example) |
| 5 | Streaming intermediate steps |
| 6 | System prompt and stopping conditions |
| 7 | Manual ReAct loop (Anthropic SDK) |
| 8 | Summary |

---
## 1. Setup

In [ ]:
# %pip install -q anthropic langchain-anthropic langgraph langchain-core python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv('.env')

import anthropic
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain.agents import create_agent

client = anthropic.Anthropic()
llm = ChatAnthropic(model="claude-sonnet-4-6")
print("Ready.")

---
## 2. First ReAct Agent with LangGraph

`create_react_agent` from LangGraph is the standard one-liner for building a ReAct agent.
Pass it a model and a list of tools — it handles the thought/action/observation loop.

The simplest possible example uses a single arithmetic tool.

In [ ]:
llm.invoke("What is 37 times 42?").content

In [ ]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


agent = create_agent(llm, [multiply])

result = agent.invoke({"messages": [{"role": "user", "content": "What is 37 times 42?"}]})
print(result["messages"][-1].content)

In [ ]:
# Inspect the full message trace to see the thought/action/observation steps
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] tool_call: {tc['name']}({tc['args']})")
    elif hasattr(msg, "content") and msg.content:
        text = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"[{role}] {text[:120]}")

---
## 3. Defining Tools with `@tool`

The `@tool` decorator converts a Python function into a LangChain tool.
The LLM uses the **function name**, **docstring**, and **type annotations** to decide
when and how to call the tool — so these must be accurate.

Three patterns for argument schemas:

| Pattern | When to use |
|---|---|
| Positional args with type hints | Simple tools with primitive args |
| Pydantic `BaseModel` | Complex or validated input schemas |
| `args_schema=` override | When docstring alone isn't enough context |

In [ ]:
from pydantic import BaseModel, Field


# Pattern 1: plain type hints
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


# Pattern 2: Pydantic schema for richer field descriptions
class GeneLookupInput(BaseModel):
    gene_symbol: str = Field(description="HGNC gene symbol, e.g. BRCA1")
    fields: list[str] = Field(
        default=["chromosome", "length_bp"],
        description="Which fields to return. Options: chromosome, length_bp, gc_content_pct",
    )


@tool(args_schema=GeneLookupInput)
def get_gene_info(gene_symbol: str, fields: list[str]) -> dict:
    """Retrieve genomic properties for a gene by its HGNC symbol."""
    db = {
        "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3},
        "EGFR":  {"chromosome": "7p11.2",   "length_bp": 188307, "gc_content_pct": 39.8},
        "KRAS":  {"chromosome": "12p12.1",  "length_bp": 45862,  "gc_content_pct": 44.6},
    }
    record = db.get(gene_symbol.upper())
    if record is None:
        return {"error": f"Gene {gene_symbol!r} not found"}
    return {k: record[k] for k in fields if k in record}


# Inspect what the LLM sees
print("Tool schema sent to LLM:")
import json
print(json.dumps(get_gene_info.args_schema.model_json_schema(), indent=2))

In [ ]:
# Test the tool directly before wiring it into an agent
print(get_gene_info.invoke({"gene_symbol": "BRCA1", "fields": ["chromosome", "gc_content_pct"]}))

---
## 4. Multi-tool Agent (Biomedical Example)

A realistic agent has multiple tools covering different domains.
The LLM selects which tool to call (or calls several in sequence) based on the query.

This example uses three mock biomedical tools — no external API keys needed.

In [ ]:
import random

GENE_DB = {
    "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1, "function": "DNA repair, tumor suppression"},
    "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3, "function": "Cell cycle regulation, apoptosis"},
    "EGFR":  {"chromosome": "7p11.2",   "length_bp": 188307,"gc_content_pct": 39.8, "function": "Cell growth signaling"},
    "KRAS":  {"chromosome": "12p12.1",  "length_bp": 45862, "gc_content_pct": 44.6, "function": "RAS/MAPK signaling"},
}

VARIANT_DB = {
    "BRCA1": [{"id": "rs80357906", "type": "missense", "clinical": "Pathogenic"},
              {"id": "rs80357713", "type": "frameshift", "clinical": "Pathogenic"}],
    "TP53":  [{"id": "rs28934578", "type": "missense", "clinical": "Pathogenic"},
              {"id": "rs1042522",  "type": "missense", "clinical": "Benign"}],
    "EGFR":  [{"id": "rs121913428","type": "missense", "clinical": "Pathogenic"}],
    "KRAS":  [{"id": "rs121913529","type": "missense", "clinical": "Pathogenic"}],
}


@tool
def get_gene_info(gene_symbol: str) -> dict:
    """Look up genomic properties (chromosome, length, GC content, function) for a gene by HGNC symbol."""
    record = GENE_DB.get(gene_symbol.upper())
    return record if record else {"error": f"Gene {gene_symbol!r} not found"}


@tool
def get_variants(gene_symbol: str) -> list[dict]:
    """Return known pathogenic and benign variants for a gene."""
    variants = VARIANT_DB.get(gene_symbol.upper())
    return variants if variants else [{"error": f"No variants found for {gene_symbol!r}"}]


@tool
def interpret_p_value(p_value: float, alpha: float = 0.05) -> dict:
    """Interpret a p-value: determine whether it is statistically significant at the given alpha level."""
    significant = p_value < alpha
    return {
        "p_value": p_value,
        "alpha": alpha,
        "significant": significant,
        "interpretation": (
            f"p={p_value} < α={alpha}: statistically significant, reject null hypothesis"
            if significant
            else f"p={p_value} ≥ α={alpha}: not significant, fail to reject null hypothesis"
        ),
    }


bio_agent = create_agent(llm, [get_gene_info, get_variants, interpret_p_value])

result = bio_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Compare BRCA1 and TP53: chromosome location and GC content. "
            "Also list pathogenic variants for each gene. "
            "Finally, is p=0.02 significant at alpha=0.05?"
        ),
    }]
})
print(result["messages"][-1].content)

In [ ]:
# Count tool calls made
from langchain_core.messages import AIMessage, ToolMessage

tool_calls = [
    tc["name"]
    for msg in result["messages"]
    if isinstance(msg, AIMessage) and msg.tool_calls
    for tc in msg.tool_calls
]
print(f"Tool calls made: {tool_calls}")

---
## 5. Streaming Intermediate Steps

By default, `agent.invoke()` returns only the final answer.
`agent.stream()` yields each step as it completes — useful for
progress feedback in long-running queries or for debugging.

`stream_mode="updates"` emits one dict per graph node update:
- `agent` node — the model's response (possibly a tool call)
- `tools` node — the tool execution result

In [ ]:
for step in bio_agent.stream(
    {"messages": [{"role": "user", "content": "What is EGFR's chromosome and any known variants?"}]},
    stream_mode="updates",
):
    node, data = next(iter(step.items()))
    for msg in data["messages"]:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[agent] → {tc['name']}({tc['args']})")
            elif msg.content:
                print(f"[agent] {msg.content}")
        elif isinstance(msg, ToolMessage):
            print(f"[tools] ← {msg.name}: {str(msg.content)[:80]}")

---
## 6. System Prompt and Stopping Conditions

Two common customizations:

**System prompt** — constrains the agent's behavior, output format, or domain.
Pass it as the `prompt` argument to `create_react_agent`.

**`recursion_limit`** — caps the number of reasoning steps to prevent infinite loops.
The default is 25; lower it for simpler tasks or raise it for complex multi-step queries.

In [ ]:
from langchain_core.messages import SystemMessage

system_prompt = SystemMessage(content=(
    "You are a concise genomics assistant. "
    "Always return answers as a bullet list. "
    "Do not include explanations beyond what is requested."
))

concise_agent = create_agent(
    model=llm,
    tools=[get_gene_info, get_variants],
    system_prompt=system_prompt,
)

result = concise_agent.invoke(
    {"messages": [{"role": "user", "content": "Give me BRCA1 and TP53 chromosome locations."}]},
    config={"recursion_limit": 10},
)
print(result["messages"][-1].content)

---
## 7. Manual ReAct Loop (Anthropic SDK)

LangChain's `create_agent` hides the loop mechanics.
Implementing it manually with the Anthropic SDK shows exactly what is happening:

1. Send messages + tools to the API
2. If `stop_reason == "tool_use"`: execute all tool calls, append results, repeat
3. If `stop_reason == "end_turn"`: return the final text

This is the same loop LangChain runs internally — useful to know when debugging.

In [ ]:
import inspect

def python_tool_to_anthropic_schema(fn) -> dict:
    """Build an Anthropic tool schema from a plain Python function."""
    sig = inspect.signature(fn)
    props = {}
    required = []
    type_map = {int: "integer", float: "number", str: "string", bool: "boolean"}
    for name, param in sig.parameters.items():
        ann = param.annotation
        json_type = type_map.get(ann, "string")
        props[name] = {"type": json_type}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    return {
        "name": fn.__name__,
        "description": (fn.__doc__ or "").strip(),
        "input_schema": {"type": "object", "properties": props, "required": required},
    }


def _get_gene_info(gene_symbol: str) -> dict:
    """Look up genomic properties for a gene by HGNC symbol."""
    return GENE_DB.get(gene_symbol.upper(), {"error": f"Gene {gene_symbol!r} not found"})


def _interpret_p_value(p_value: float, alpha: float = 0.05) -> dict:
    """Interpret a p-value: statistically significant at the given alpha level?"""
    significant = p_value < alpha
    return {
        "p_value": p_value,
        "alpha": alpha,
        "significant": significant,
    }


TOOL_REGISTRY = {
    "_get_gene_info": _get_gene_info,
    "_interpret_p_value": _interpret_p_value,
}

ANTHROPIC_TOOLS = [
    python_tool_to_anthropic_schema(_get_gene_info),
    python_tool_to_anthropic_schema(_interpret_p_value),
]

print("Anthropic tool schemas:")
for t in ANTHROPIC_TOOLS:
    print(f"  {t['name']}: {t['description']}")

In [ ]:
def anthropic_react_agent(query: str, tools: list[dict], registry: dict, verbose: bool = True) -> str:
    """Manual ReAct loop using the Anthropic SDK."""
    messages = [{"role": "user", "content": query}]
    step = 0

    while True:
        step += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if hasattr(b, "text")), "")

        # append the model's response (may contain tool_use blocks)
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                fn = registry[block.name]
                output = fn(**block.input)
                if verbose:
                    print(f"  step {step}: {block.name}({block.input}) → {output}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output),
                })

        messages.append({"role": "user", "content": tool_results})


answer = anthropic_react_agent(
    "What chromosome is KRAS on, and is p=0.001 significant at alpha=0.01?",
    ANTHROPIC_TOOLS,
    TOOL_REGISTRY,
)
print("\nFinal answer:")
print(answer)

### Parallel tool calls

Claude can emit multiple `tool_use` blocks in a single response when the calls
are independent — this halves latency compared to sequential calls.
The loop above already handles this: it collects *all* `tool_use` blocks
from one response before appending results.

In [ ]:
# A query that triggers parallel tool calls in a single model turn
answer = anthropic_react_agent(
    "Look up BRCA1 and TP53 gene info at the same time, then tell me which has higher GC content.",
    ANTHROPIC_TOOLS,
    TOOL_REGISTRY,
)
print("\nFinal answer:")
print(answer)

---
## 8. Summary

| Approach | Lines of code | Visibility | When to use |
|---|---|---|---|
| `create_react_agent` (LangGraph) | ~5 | Step-level via `stream()` | Standard use; integrates with LangChain ecosystem |
| Manual loop (Anthropic SDK) | ~30 | Full | Need to inspect every message; custom routing logic |

**Key patterns**

- **`@tool` decorator** — converts any Python function to a tool; docstring and type hints become the schema.
- **Pydantic `args_schema`** — use when tool arguments need rich descriptions or validation.
- **`stream_mode="updates"`** — exposes each thought/action/observation step for debugging or progress display.
- **System prompt** — constrain output format, domain, or verbosity via `prompt=` in `create_react_agent`.
- **Parallel tool calls** — Claude batches independent calls in one turn automatically; your loop must handle multiple `tool_use` blocks per response.

**Next steps**

- `mcp_agents.ipynb` — bind MCP servers to provide tools from external processes
- `python_agents_sdk.ipynb` — managed agent loops with the Anthropic Agents SDK